In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

## Question 9.1

In [11]:
n = 1000
simulations = 1000

coefs = []
ols_se = []
robust_se = []

for i in range(simulations):

    X = np.random.uniform(1, 10, n)
    error = np.random.normal(0, 0.5 * X, n)

    Y = 4 + 7 * X + error

    X = sm.add_constant(X)

    model = sm.OLS(Y, X).fit()

    coefs.append(model.params[1])
    ols_se.append(model.bse[1])

simulation_sd = np.std(coefs)

average_ols_se = np.mean(ols_se)
average_robust_se = np.mean(robust_se)

print(f'Simulation Standard Deviation: {simulation_sd:.3f}')
print(f'Average OLS Standard Error: {average_ols_se:.3f}')

Simulation Standard Deviation: 0.040
Average OLS Standard Error: 0.037


## Question 9.2

In [21]:
n = 1000
simulations = 1000
rho = 0.9

X = np.random.normal(0, 1, n)

X_const = sm.add_constant(X)

coefs = []
ols_se = []

for i in range(simulations): 

    u = np.random.normal(0, 1, n)
    error = np.zeros(n)

    for t in range(1, n):
        error[t] = rho * error[t-1] + u[t]

    Y = 4 + 7 * X + error

    model = sm.OLS(Y, X_const).fit()

    coefs.append(model.params[1])
    ols_se.append(model.bse[1])

simulation_sd = np.std(coefs)
average_ols_se = np.mean(ols_se)

print(f'Simulation Standard Deviation: {simulation_sd:.3f}')
print(f'Average OLS Standard Error: {average_ols_se:.3f}')

Simulation Standard Deviation: 0.078
Average OLS Standard Error: 0.071


In [24]:
u = np.random.normal(0, 1, n)
error = np.zeros(n)

error[0] = np.random.normal(0, 1)

for t in range(1, n):
    error[t] = rho * error[t-1] + u[t]

Y = 4 + 7 * X + error

data = pd.DataFrame({'X' : X, 'Y' : Y})

bootstrap_coefs = []

for i in range(1000):

    bootstrap_data = data.sample(n = n, replace = True)

    X_boot = sm.add_constant(bootstrap_data['X'])

    model = sm.OLS(bootstrap_data['Y'], X_boot).fit()

    bootstrap_coefs.append(model.params['X'])

bootstrap_sd = np.std(bootstrap_coefs)

print(f'Bootstrap Standard Deviation: {bootstrap_sd:.3f}')

Bootstrap Standard Deviation: 0.064


## Question 11.1

In [ ]:
n = 100
time = np.arange(n)

event_time = 50

post = (time >= event_time).astype(int)

error = np.random.normal(0, 2, n)

Y = 12 + 0.4 * time + 0.02 * time**2 + 6 * post + 0.3 * (time - event_time) * post + 0.03 * (time - event_time)**2 * post + error

data = pd.DataFrame({'time' : time, 'post' : post, 'Y' : Y})

data.head()

,time,post,Y
0,0,0,11.489964
1,1,0,13.177907
2,2,0,13.405158
3,3,0,13.799013
4,4,0,14.189996


In [26]:
X1 = data['post']
X1 = sm.add_constant(X1)

model1 = sm.OLS(data['Y'], X1).fit()

model1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      Y   R-squared:                       0.672
Model:                            OLS   Adj. R-squared:                  0.669
Method:                 Least Squares   F-statistic:                     200.7
Date:                Tue, 24 Nov 2026   Prob (F-statistic):           1.89e-25
Time:                        16:09:13   Log-Likelihood:                -542.58
No. Observations:                 100   AIC:                             1089.
Df Residuals:                      98   BIC:                             1094.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         37.5173      7.854      4.777      0.000      21.932      53.103
post         157.3325     11.107     14.166      0.000     135.292     179.373
==============================================================================
Omnibus:                        4.017   Durbin-Watson:                   0.079
Prob(Omnibus):                  0.134   Jarque-Bera (JB):                3.398
Skew:                           0.429   Prob(JB):                        0.183
Kurtosis:                       3.284   Cond. No.                         2.62
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [27]:
data['time_sq'] = data['time']**2
data['time_after'] = (data['time'] - event_time) * data['post']
data['time_after_sq'] = (data['time'] - event_time)**2 * data['post']

X2 = data[['time', 'time_sq', 'post', 'time_after', 'time_after_sq']]
X2 = sm.add_constant(X2)

model2 = sm.OLS(data['Y'], X2).fit()

model2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      Y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 4.227e+04
Date:                Tue, 24 Nov 2026   Prob (F-statistic):          7.11e-156
Time:                        16:11:13   Log-Likelihood:                -212.37
No. Observations:                 100   AIC:                             436.7
Df Residuals:                      94   BIC:                             452.4
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            11.6450      0.851     13.681      0.000       9.955      13.335
time              0.3709      0.080      4.617      0.000       0.211       0.530
time_sq           0.0208      0.002     13.093      0.000       0.018       0.024
post              6.0081      1.255      4.788      0.000       3.517       8.500
time_after        0.2648      0.116      2.286      0.024       0.035       0.495
time_after_sq     0.0291      0.002     12.980      0.000       0.025       0.034
==============================================================================
Omnibus:                        0.264   Durbin-Watson:                   2.196
Prob(Omnibus):                  0.876   Jarque-Bera (JB):                0.443
Skew:                           0.012   Prob(JB):                        0.801
Kurtosis:                       2.675   Cond. No.                     2.75e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.75e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [30]:
simulations = 1000

model1_coefs = []
model2_coefs = []

for i in range(1000):

    error = np.random.normal(0, 2, n)

    Y = 12 + 0.4 * time + 0.02 * time**2 + 6 * post + 0.3 * (time - event_time) * post + 0.03 * (time - event_time)**2 * post + error

    data = pd.DataFrame({'time' : time, 'post' : post, 'Y' : Y})

    X1 = sm.add_constant(data['post'])
    model1 = sm.OLS(Y, X1).fit()

    model1_coefs.append(model1.params['post'])

    data['time_sq'] = time**2
    data['time_after'] = (time - event_time) * post
    data['time_after_sq'] = (time - event_time)**2 * post

    X2 = data[['time', 'time_sq', 'post', 'time_after', 'time_after_sq']]
    X2 = sm.add_constant(X2)

    model2 = sm.OLS(Y, X2).fit()
    model2_coefs.append(model2.params[['post', 'time_after', 'time_after_sq']])

model1_average = np.mean(model1_coefs)
model2_average = pd.DataFrame(model2_coefs).mean()

print(f'Model 1 Average Post: {model1_average:.3f}')
print(model2_average)

Model 1 Average Post: 156.610
post             5.991823
time_after       0.297517
time_after_sq    0.029985
dtype: float64


## Question 11.2

In [31]:
time = np.arange(100)
event_time = 50

groups = ['A', 'B', 'C']

starting_values = {'A' : 12, 'B': 18, 'C' : 24}

data = []

for group in groups:
    for t in time: 
        post = (t >= event_time).astype(int)

        y = (starting_values[group] + 0.5 * t + 7 * post + np.random.normal(0, 1))

        data.append([group, t, post, y])

df = pd.DataFrame(data, columns = ['group', 'time', 'post', 'Y'])

df.head()

,group,time,post,Y
0,A,0,0,12.885306
1,A,1,0,12.363357
2,A,2,0,12.938210
3,A,3,0,13.954837
4,A,4,0,13.987297


In [32]:
X = pd.get_dummies(df[['group']], drop_first = True, dtype = int)

X['time'] = df['time']
X['post'] = df['post']

X = sm.add_constant(X)

Y = df['Y']

model = sm.OLS(Y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.997
Model:                            OLS   Adj. R-squared:                  0.997
Method:                 Least Squares   F-statistic:                 2.709e+04
Date:                Tue, 24 Nov 2026   Prob (F-statistic):               0.00
Time:                        16:49:57   Log-Likelihood:                -409.71
No. Observations:                 300   AIC:                             829.4
Df Residuals:                     295   BIC:                             847.9
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         11.9624      0.145     82.600      0.0